# Embedding Similarity

This notebook separates two questions that are easy to confuse:

1. **Local representation:** inside one UniEdit record, does an embedding place an edit near its generality prompts and away from locality prompts? This is a diagnostic of the representation itself. It does not build a memory bank or choose an activation threshold.
2. **Global routing transfer:** after thresholds are tuned only on UniEdit, do frozen thresholds retrieve and activate the correct edit in seeded CounterFact banks? This tests whether routing behavior generalizes to held-out evaluation data instead of being tuned directly on CounterFact.

Local mode reports edit-generality, edit-locality, and generality-locality similarity distributions plus the rate at which generality is preferred over locality. Global mode calibrates a score or score-plus-margin gate on a UniEdit calibration split, checks it on held-out UniEdit prompts, freezes it, and then reports CounterFact Recall@1, MRR, correct routing, positive rejection, incorrect routing, locality false-activation rate, processing time, throughput, and peak CUDA memory at each bank size.

High retrieval with poor routing means the correct edit is ranked first but the threshold rejects it. High correct routing with high locality false activation means the gate is too permissive. Stable CounterFact results close to held-out UniEdit results indicate cross-dataset transfer; a large drop indicates that the encoder or thresholds overfit UniEdit. These experiments test semantic routing only, not whether a retrieved edit can solve multi-hop causal reasoning.

## Environment

In [ ]:
import gc
import hashlib
import json
import sys
import time
import unicodedata
from collections.abc import Callable, Mapping, Sequence
from contextlib import redirect_stderr
from dataclasses import dataclass, field
from io import StringIO
from pathlib import Path
from random import Random
from time import perf_counter
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from datasets import load_dataset, load_from_disk
from IPython.display import display
from matplotlib.ticker import MaxNLocator
from tqdm.auto import tqdm


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "notebook").exists():
            return path
    raise RuntimeError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_REPO = "Savassan/KnowledgeSurgeon"
DATASET_CONFIG = "similarity"
COUNTERFACT_DATASET_CONFIG = "es_loc"
DATASET_SPLIT = "data"
LOCAL_DATASET_PATH = PROJECT_ROOT / "data" / "datasets" / "uniedit" / "final"
LOCAL_COUNTERFACT_PATH = PROJECT_ROOT / "data" / "datasets" / "counterfact" / "final"
EVAL_CONFIG_PATH = PROJECT_ROOT / "src" / "evaluate" / "output.yaml"
OUTPUT_DIR = PROJECT_ROOT / "data" / "similarity"
REPRESENTATION_TYPE = "normalized_dense_sentence_embedding"

EMBEDDING_MODEL_SPECS = {
    "all-mpnet-base-v2": {
        "repo": "sentence-transformers/all-mpnet-base-v2",
        "backend": "sentence_transformers",
        "prefixes": {"symmetric": "", "query": "", "passage": ""},
    },
    "all-MiniLM-L6-v2": {
        "repo": "sentence-transformers/all-MiniLM-L6-v2",
        "backend": "sentence_transformers",
        "prefixes": {"symmetric": "", "query": "", "passage": ""},
    },
    "all-MiniLM-L12-v2": {
        "repo": "sentence-transformers/all-MiniLM-L12-v2",
        "backend": "sentence_transformers",
        "prefixes": {"symmetric": "", "query": "", "passage": ""},
    },
    "e5-base-v2": {
        "repo": "intfloat/e5-base-v2",
        "backend": "sentence_transformers",
        "prefixes": {
            "symmetric": "query: ",
            "query": "query: ",
            "passage": "passage: ",
        },
    },
    "e5-large-v2": {
        "repo": "intfloat/e5-large-v2",
        "backend": "sentence_transformers",
        "prefixes": {
            "symmetric": "query: ",
            "query": "query: ",
            "passage": "passage: ",
        },
    },
    "bge-base-en-v1.5": {
        "repo": "BAAI/bge-base-en-v1.5",
        "backend": "sentence_transformers",
        "prefixes": {"symmetric": "", "query": "", "passage": ""},
    },
    "bge-large-en-v1.5": {
        "repo": "BAAI/bge-large-en-v1.5",
        "backend": "sentence_transformers",
        "prefixes": {"symmetric": "", "query": "", "passage": ""},
    },
    "bge-m3": {
        "repo": "BAAI/bge-m3",
        "backend": "flag_embedding",
        "prefixes": {"symmetric": "", "query": "", "passage": ""},
    },
}


def resolve_embedding_model_keys(model_keys: list[str]) -> list[str]:
    selected = list(dict.fromkeys(model_keys))
    if not selected:
        raise ValueError("embedding_model_keys must contain at least one model.")
    unknown = [key for key in selected if key not in EMBEDDING_MODEL_SPECS]
    if unknown:
        raise ValueError(f"Unknown embedding model keys: {unknown}")
    return selected


def local_output_path(
    model_key: str,
    record_limit: int | None,
) -> Path:
    if record_limit is not None and int(record_limit) <= 0:
        raise ValueError("record_limit must be positive when set.")
    output_stem = model_key if record_limit is None else f"{model_key}.limit-{int(record_limit)}"
    return OUTPUT_DIR / "local" / embedding_precision_mode() / f"{output_stem}.json"


print(f"local_dataset={LOCAL_DATASET_PATH}")
print(f"output_dir={OUTPUT_DIR}")


## Embedding Backends

In [ ]:
def embedding_device() -> str:
    return f"cuda:{EMBEDDING_GPU_ID}" if torch.cuda.is_available() else "cpu"


def embedding_compute_dtype() -> str:
    return "float16" if EMBEDDING_USE_FP16 and torch.cuda.is_available() else "float32"


def embedding_precision_mode() -> str:
    return "fp16" if embedding_compute_dtype() == "float16" else "fp32"


def load_embedding_model(model_key: str):
    spec = EMBEDDING_MODEL_SPECS[model_key]
    device = embedding_device()
    if spec["backend"] == "sentence_transformers":
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer(spec["repo"], device=device)
        model.max_seq_length = min(int(model.max_seq_length), EMBEDDING_MAX_LENGTH)
        if embedding_compute_dtype() == "float16":
            model.half()
    elif spec["backend"] == "flag_embedding":
        from FlagEmbedding import BGEM3FlagModel

        model = BGEM3FlagModel(
            spec["repo"],
            use_fp16=embedding_compute_dtype() == "float16",
            devices=device,
        )
    else:
        raise ValueError(f"Unsupported embedding backend: {spec['backend']}")
    print(
        f"loaded_embedding_model={model_key} repo={spec['repo']} "
        f"backend={spec['backend']} device={device} "
        f"compute_dtype={embedding_compute_dtype()}"
    )
    return model


def embedding_effective_max_length(model, model_key: str) -> int:
    if EMBEDDING_MODEL_SPECS[model_key]["backend"] == "sentence_transformers":
        return int(model.max_seq_length)
    return EMBEDDING_MAX_LENGTH


def format_embedding_prompts(
    model_key: str,
    prompts: list[str],
    prompt_role: str,
) -> list[str]:
    prefixes = EMBEDDING_MODEL_SPECS[model_key]["prefixes"]
    if prompt_role not in prefixes:
        raise ValueError(f"Unsupported prompt role: {prompt_role}")
    prefix = prefixes[prompt_role]
    return [f"{prefix}{prompt}" for prompt in prompts]


def encode_flag_embedding_prompts(model, prompts: list[str]) -> dict[str, Any]:
    with redirect_stderr(StringIO()):
        return model.encode(
            prompts,
            batch_size=EMBEDDING_BATCH_SIZE,
            max_length=EMBEDDING_MAX_LENGTH,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False,
        )


def encode_prompts(
    model,
    model_key: str,
    prompts: list[str],
    prompt_role: str,
) -> np.ndarray:
    if not prompts:
        raise ValueError("Cannot encode an empty prompt list.")
    spec = EMBEDDING_MODEL_SPECS[model_key]
    formatted_prompts = format_embedding_prompts(model_key, prompts, prompt_role)
    if spec["backend"] == "sentence_transformers":
        embeddings = model.encode(
            formatted_prompts,
            batch_size=EMBEDDING_BATCH_SIZE,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
    else:
        encoded = encode_flag_embedding_prompts(model, formatted_prompts)
        embeddings = encoded["dense_vecs"]

    embeddings = np.asarray(embeddings, dtype=np.float32)
    norms = np.linalg.norm(embeddings, axis=-1, keepdims=True)
    embeddings = embeddings / np.clip(norms, 1e-12, None)
    if embeddings.ndim != 2 or embeddings.shape[0] != len(prompts):
        raise ValueError(
            f"Unexpected embedding shape for {model_key}: {embeddings.shape}; "
            f"expected ({len(prompts)}, hidden_size)."
        )
    return embeddings


def move_embedding_model_to_cpu(model, model_key: str) -> None:
    backend = EMBEDDING_MODEL_SPECS[model_key]["backend"]
    try:
        if backend == "sentence_transformers":
            model.to("cpu")
        else:
            underlying_model = getattr(model, "model", None)
            if underlying_model is not None and hasattr(underlying_model, "to"):
                underlying_model.to("cpu")
    except Exception as error:
        print(f"embedding_model_cpu_move_warning={model_key}: {error}")


def clear_embedding_memory(model_key: str) -> None:
    gc.collect()
    if not torch.cuda.is_available():
        return
    try:
        torch.cuda.synchronize(EMBEDDING_GPU_ID)
    except Exception:
        pass
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass
    allocated = torch.cuda.memory_allocated(EMBEDDING_GPU_ID) / 1024**3
    reserved = torch.cuda.memory_reserved(EMBEDDING_GPU_ID) / 1024**3
    print(
        f"released_embedding_model={model_key} "
        f"cuda_allocated_gb={allocated:.3f} cuda_reserved_gb={reserved:.3f}"
    )

## Record Construction

In [ ]:
PAIR_AGGREGATION = "all_prompt_pairs"
PAIR_SPECS = (
    ("p1-p2", "edit", "generality", "edit - generality"),
    ("p1-p3", "edit", "locality", "edit - locality"),
    ("p2-p3", "generality", "locality", "generality - locality"),
)


def load_similarity_dataset(dataset_source: str):
    if dataset_source == "local":
        return load_from_disk(str(LOCAL_DATASET_PATH))["similarity"]
    if dataset_source == "huggingface":
        return load_dataset(DATASET_REPO, DATASET_CONFIG, split=DATASET_SPLIT)
    raise ValueError("dataset_source must be 'local' or 'huggingface'.")


def source_record_count(dataset, record_limit: int | None) -> int:
    if record_limit is None:
        return len(dataset)
    if int(record_limit) <= 0:
        raise ValueError("record_limit must be positive when set.")
    return min(int(record_limit), len(dataset))


def build_local_similarity_records(
    dataset,
    record_limit: int | None,
) -> list[dict[str, Any]]:
    records = []
    for source_index in range(source_record_count(dataset, record_limit)):
        source_record = dataset[source_index]
        role_prompts = {
            "edit": [source_record["edit"]["prompt"]],
            "generality": list(source_record["generality"]),
            "locality": [item["prompt"] for item in source_record["locality"]],
        }
        for role, prompts in role_prompts.items():
            if not prompts or any(not prompt for prompt in prompts):
                raise ValueError(f"Empty {role} prompt list at source index {source_index}.")
        records.append({"source_index": source_index, "role_prompts": role_prompts})
    return records


## Similarity Calculation

In [ ]:
def analyze_local_record_batch(
    record_batch: list[dict[str, Any]],
    model,
    model_key: str,
) -> list[dict[str, Any]]:
    prompts = [
        prompt
        for record in record_batch
        for role_prompts in record["role_prompts"].values()
        for prompt in role_prompts
    ]
    embeddings = encode_prompts(model, model_key, prompts, prompt_role="symmetric")

    analyzed_records = []
    offset = 0
    for record in record_batch:
        role_embeddings = {}
        prompt_counts = {}
        for role, role_prompts in record["role_prompts"].items():
            count = len(role_prompts)
            role_embeddings[role] = embeddings[offset : offset + count]
            prompt_counts[role] = count
            offset += count

        pair_values = {}
        for pair_key, left_role, right_role, _ in PAIR_SPECS:
            matrix = role_embeddings[left_role] @ role_embeddings[right_role].T
            pair_values[pair_key] = [
                round(float(value), 8) for value in matrix.reshape(-1)
            ]
        analyzed_records.append({
            "source_index": record["source_index"],
            "prompt_counts": prompt_counts,
            "pair_values": pair_values,
        })
    return analyzed_records


def aggregate_local_records(records: list[dict[str, Any]]) -> dict[str, Any]:
    if not records:
        raise ValueError("No records were analyzed.")

    pairs = {}
    for pair_key, _, _, label in PAIR_SPECS:
        values = np.asarray(
            [
                value
                for record in records
                for value in record["pair_values"][pair_key]
            ],
            dtype=np.float64,
        )
        if values.size == 0:
            raise ValueError(f"No values were produced for pair {pair_key}.")
        pairs[pair_key] = {
            "label": label,
            "count": int(values.size),
            "mean": round(float(values.mean()), 8),
            "std": round(float(values.std(ddof=0)), 8),
        }
    return {"record_count": len(records), "pairs": pairs}


def write_json_atomic(payload: dict, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(f"{destination.suffix}.tmp")
    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, ensure_ascii=True, allow_nan=False)
    temporary.replace(destination)


def similarity_prompt_fingerprint(dataset) -> str:
    digest = hashlib.sha256()
    for record in dataset:
        prompt_record = {
            "edit": record["edit"]["prompt"],
            "generality": list(record["generality"]),
            "locality": [item["prompt"] for item in record["locality"]],
        }
        encoded = json.dumps(
            prompt_record,
            ensure_ascii=False,
            separators=(",", ":"),
        ).encode("utf-8")
        digest.update(len(encoded).to_bytes(8, "big"))
        digest.update(encoded)
    return digest.hexdigest()


def similarity_dataset_metadata(dataset, dataset_source: str) -> dict[str, Any]:
    metadata = {
        "source": dataset_source,
        "rows": len(dataset),
        "fingerprint": getattr(dataset, "_fingerprint", None),
        "prompt_fingerprint": similarity_prompt_fingerprint(dataset),
    }
    if dataset_source == "local":
        metadata["dataset_id"] = "uniedit/final"
    else:
        metadata.update({
            "repo": DATASET_REPO,
            "config": DATASET_CONFIG,
            "split": DATASET_SPLIT,
        })
    return metadata


def run_local_similarity_analysis(
    dataset,
    records: list[dict[str, Any]],
    model,
    model_key: str,
    output_path: Path,
    record_batch_size: int,
    dataset_metadata: dict[str, Any],
    experiment: dict[str, Any],
) -> None:
    if record_batch_size <= 0:
        raise ValueError("record_batch_size must be positive.")

    completed_records = []
    for start in tqdm(
        range(0, len(records), record_batch_size),
        desc=f"Local embeddings [{model_key}]",
    ):
        completed_records.extend(
            analyze_local_record_batch(
                records[start : start + record_batch_size],
                model=model,
                model_key=model_key,
            )
        )

    completed_records.sort(key=lambda record: record["source_index"])
    result = build_result_header(
        model=model,
        model_key=model_key,
        dataset_metadata=dataset_metadata,
        experiment=experiment,
    )
    result.update({
        "pair_aggregation": PAIR_AGGREGATION,
        "pair_order": [
            {"pair": pair_key, "label": label}
            for pair_key, _, _, label in PAIR_SPECS
        ],
        "records": completed_records,
        "aggregate": aggregate_local_records(completed_records),
    })
    write_json_atomic(result, output_path)
    print(f"saved={output_path} records={len(completed_records)}")


## Global Retrieval Calculation

Global mode is implemented directly here so the complete UniEdit calibration, frozen CounterFact evaluation, timing, diagnostics, and result aggregation pipeline remains visible in the notebook.

In [ ]:

def _prompt_key(prompt: str) -> str:
    normalized = unicodedata.normalize("NFKC", prompt).casefold()
    characters = [
        character
        if character.isalnum() or unicodedata.category(character).startswith("S")
        else " "
        for character in normalized
    ]
    return " ".join("".join(characters).split())


def _dataset_fingerprint(records: Sequence[Mapping[str, Any]]) -> str:
    digest = hashlib.sha256()
    for record in records:
        payload = {
            "edit": record["edit_prompt"],
            "positive": list(record["positive_prompts"]),
            "locality": list(record["locality_prompts"]),
        }
        encoded = json.dumps(
            payload,
            ensure_ascii=False,
            separators=(",", ":"),
        ).encode("utf-8")
        digest.update(len(encoded).to_bytes(8, "big"))
        digest.update(encoded)
    return digest.hexdigest()


def dataset_identity(
    records: Sequence[Mapping[str, Any]],
) -> dict[str, Any]:
    return {
        "rows": len(records),
        "prompt_fingerprint": _dataset_fingerprint(records),
    }


def build_uniedit_records(dataset, record_limit: int | None = None) -> list[dict[str, Any]]:
    count = len(dataset) if record_limit is None else min(int(record_limit), len(dataset))
    if count <= 0:
        raise ValueError("UniEdit calibration requires at least one record.")
    records = []
    for source_index in range(count):
        source = dataset[source_index]
        record = {
            "source_index": source_index,
            "edit_prompt": source["edit"]["prompt"],
            "positive_prompts": list(source["generality"]),
            "locality_prompts": [item["prompt"] for item in source["locality"]],
        }
        _validate_record(record, "calibration")
        records.append(record)
    return records


def build_counterfact_records(
    dataset,
    record_limit: int | None = None,
) -> list[dict[str, Any]]:
    count = len(dataset) if record_limit is None else min(int(record_limit), len(dataset))
    if count <= 0:
        raise ValueError("CounterFact testing requires at least one record.")
    records = []
    for source_index in range(count):
        source = dataset[source_index]
        record = {
            "source_index": source_index,
            "edit_prompt": source["edit"]["prompt"],
            "positive_prompts": list(source["generality"]),
            "locality_prompts": [item["prompt"] for item in source["locality"]],
        }
        _validate_record(record, "test")
        records.append(record)
    return records


def _validate_record(record: Mapping[str, Any], label: str) -> None:
    if not record["edit_prompt"]:
        raise ValueError(f"Empty {label} edit prompt at row {record['source_index']}.")
    if not record["positive_prompts"] or any(
        not prompt for prompt in record["positive_prompts"]
    ):
        raise ValueError(f"Empty {label} positive prompt at row {record['source_index']}.")
    if not record["locality_prompts"] or any(
        not prompt for prompt in record["locality_prompts"]
    ):
        raise ValueError(f"Empty {label} locality prompt at row {record['source_index']}.")


def _validate_settings(settings: Mapping[str, Any], calibration_size: int, test_size: int) -> None:
    bank_sizes = [int(size) for size in settings["bank_sizes"]]
    if not bank_sizes or any(size < 2 for size in bank_sizes):
        raise ValueError("Global bank sizes must contain integers above one.")
    if bank_sizes != sorted(set(bank_sizes)):
        raise ValueError("Global bank sizes must be unique and increasing.")
    if bank_sizes[-1] > min(calibration_size, test_size):
        raise ValueError(
            "The largest global bank exceeds the calibration or test dataset."
        )
    if int(settings["n_runs"]) <= 0:
        raise ValueError("Global n_runs must be positive.")
    if int(settings["query_owner_count"]) < 2:
        raise ValueError("Global query_owner_count must be at least two.")
    if int(settings["query_owner_count"]) > bank_sizes[0]:
        raise ValueError("Global query_owner_count exceeds the smallest bank.")
    fraction = float(settings["calibration_fraction"])
    if not 0.0 < fraction < 1.0:
        raise ValueError("Global calibration_fraction must be in (0, 1).")
    far_target = float(settings["locality_far_target"])
    if not 0.0 < far_target < 1.0:
        raise ValueError("Global locality_far_target must be in (0, 1).")
    if settings["gate"] not in {"score", "score_margin"}:
        raise ValueError("Global gate must be 'score' or 'score_margin'.")
    if int(settings["retrieval_batch_size"]) <= 0:
        raise ValueError("Global retrieval_batch_size must be positive.")
    retrieval_device = str(settings["retrieval_device"])
    if retrieval_device != "cpu" and not retrieval_device.startswith("cuda"):
        raise ValueError("Global retrieval_device must be 'cpu' or a CUDA device.")
    if retrieval_device.startswith("cuda") and not torch.cuda.is_available():
        raise ValueError("CUDA retrieval was requested but CUDA is unavailable.")


def _shuffle_indices(size: int, seed: int) -> np.ndarray:
    values = list(range(size))
    Random(seed).shuffle(values)
    return np.asarray(values, dtype=np.int64)


def _sample_indices(size: int, limit: int | None, seed: int) -> np.ndarray:
    if limit is None or size <= int(limit):
        return np.arange(size, dtype=np.int64)
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(size, size=int(limit), replace=False)).astype(np.int64)


def _derived_seed(base_seed: int, *parts: Any) -> int:
    payload = "|".join(str(part) for part in (base_seed, *parts)).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "little")


def _unique_prompts(
    records: Sequence[Mapping[str, Any]],
    indices: Sequence[int],
    field: str,
) -> list[dict[str, Any]]:
    prompts: dict[str, dict[str, Any]] = {}
    for position in indices:
        record = records[int(position)]
        for prompt in record[field]:
            prompts.setdefault(
                _prompt_key(prompt),
                {
                    "prompt": prompt,
                    "source_index": int(record["source_index"]),
                },
            )
    return list(prompts.values())


def _owner_queries(
    records: Sequence[Mapping[str, Any]],
    bank_indices: np.ndarray,
    owner_positions: np.ndarray,
) -> list[dict[str, Any]]:
    queries = []
    for owner_position in owner_positions:
        record_index = int(bank_indices[int(owner_position)])
        record = records[record_index]
        queries.extend({
            "prompt": prompt,
            "owner_position": int(owner_position),
            "source_index": int(record["source_index"]),
        } for prompt in record["positive_prompts"])
    return queries


@dataclass
class TimingLedger:
    seconds: dict[str, float] = field(default_factory=dict)
    counts: dict[str, int] = field(default_factory=dict)

    def add(self, key: str, seconds: float, count: int = 0) -> None:
        self.seconds[key] = self.seconds.get(key, 0.0) + float(seconds)
        if count:
            self.counts[key] = self.counts.get(key, 0) + int(count)

    def as_dict(self) -> dict[str, Any]:
        throughput = {
            key: round(self.counts[key] / seconds, 4)
            for key, seconds in self.seconds.items()
            if seconds > 0 and key in self.counts
        }
        return {
            "seconds": {
                key: round(value, 6) for key, value in self.seconds.items()
            },
            "counts": dict(self.counts),
            "items_per_second": throughput,
        }


def _sync_device(device: str) -> None:
    if not device.startswith("cuda") or not torch.cuda.is_available():
        return
    torch.cuda.synchronize(torch.device(device))


def _timed_call(
    ledger: TimingLedger,
    key: str,
    device: str,
    function: Callable[[], Any],
    *,
    count: int = 0,
) -> Any:
    _sync_device(device)
    start = time.perf_counter()
    result = function()
    _sync_device(device)
    ledger.add(key, time.perf_counter() - start, count=count)
    return result


def _timed_encode(
    encode_prompts: Callable[..., np.ndarray],
    model,
    model_key: str,
    prompts: list[str],
    prompt_role: str,
    ledger: TimingLedger,
    timing_key: str,
    device: str,
) -> np.ndarray:
    return _timed_call(
        ledger,
        timing_key,
        device,
        lambda: encode_prompts(model, model_key, prompts, prompt_role=prompt_role),
        count=len(prompts),
    )


def _positive_retrieval(
    query_embeddings: np.ndarray,
    bank_embeddings: np.ndarray,
    owner_positions: np.ndarray,
    batch_size: int,
    device: str,
) -> dict[str, np.ndarray]:
    if len(query_embeddings) != len(owner_positions):
        raise ValueError("Each positive query must have an owner position.")
    bank = torch.as_tensor(bank_embeddings, dtype=torch.float32, device=device)
    outputs: dict[str, list[np.ndarray]] = {
        "ranks": [],
        "top1_scores": [],
        "top2_scores": [],
        "top1_positions": [],
        "correct_scores": [],
        "best_negative_scores": [],
        "top1_correct": [],
    }
    with torch.inference_mode():
        for start in range(0, len(query_embeddings), batch_size):
            stop = min(start + batch_size, len(query_embeddings))
            queries = torch.as_tensor(
                query_embeddings[start:stop],
                dtype=torch.float32,
                device=device,
            )
            owners = torch.as_tensor(
                owner_positions[start:stop],
                dtype=torch.long,
                device=device,
            )
            scores = queries @ bank.T
            rows = torch.arange(len(owners), device=device)
            correct = scores[rows, owners]
            top_scores, top_positions = torch.topk(scores, k=2, dim=1)
            top1_correct = top_positions[:, 0].eq(owners)
            best_negative = torch.where(top1_correct, top_scores[:, 1], top_scores[:, 0])
            outputs["ranks"].append(
                (scores > correct[:, None]).sum(dim=1).add(1).cpu().numpy()
            )
            outputs["top1_scores"].append(top_scores[:, 0].cpu().numpy())
            outputs["top2_scores"].append(top_scores[:, 1].cpu().numpy())
            outputs["top1_positions"].append(top_positions[:, 0].cpu().numpy())
            outputs["correct_scores"].append(correct.cpu().numpy())
            outputs["best_negative_scores"].append(best_negative.cpu().numpy())
            outputs["top1_correct"].append(top1_correct.cpu().numpy())
    result = {key: np.concatenate(values) for key, values in outputs.items()}
    result["top1_margins"] = result["top1_scores"] - result["top2_scores"]
    return result


def _locality_retrieval(
    query_embeddings: np.ndarray,
    bank_embeddings: np.ndarray,
    batch_size: int,
    device: str,
) -> dict[str, np.ndarray]:
    bank = torch.as_tensor(bank_embeddings, dtype=torch.float32, device=device)
    outputs: dict[str, list[np.ndarray]] = {
        "top1_scores": [],
        "top2_scores": [],
        "top1_positions": [],
    }
    with torch.inference_mode():
        for start in range(0, len(query_embeddings), batch_size):
            queries = torch.as_tensor(
                query_embeddings[start:start + batch_size],
                dtype=torch.float32,
                device=device,
            )
            scores = queries @ bank.T
            top_scores, top_positions = torch.topk(scores, k=2, dim=1)
            outputs["top1_scores"].append(top_scores[:, 0].cpu().numpy())
            outputs["top2_scores"].append(top_scores[:, 1].cpu().numpy())
            outputs["top1_positions"].append(top_positions[:, 0].cpu().numpy())
    result = {key: np.concatenate(values) for key, values in outputs.items()}
    result["top1_margins"] = result["top1_scores"] - result["top2_scores"]
    return result


def _concatenate(results: Sequence[Mapping[str, np.ndarray]]) -> dict[str, np.ndarray]:
    if not results:
        raise ValueError("Cannot concatenate an empty retrieval collection.")
    keys = set.intersection(*(set(result) for result in results))
    return {
        key: np.concatenate([np.asarray(result[key]) for result in results])
        for key in keys
    }


def _maximum_false_activations(query_count: int, target: float) -> int:
    return int(np.floor(query_count * target + 1e-12))


def _score_threshold_for_budget(
    scores: np.ndarray,
    eligible: np.ndarray,
    maximum_active: int,
) -> float:
    eligible_scores = scores[eligible]
    if eligible_scores.size <= maximum_active:
        return float(np.nextafter(-1.0, -np.inf))
    ordered = np.sort(eligible_scores)[::-1]
    return float(np.nextafter(ordered[maximum_active], np.inf))


def _active_mask(
    retrieval: Mapping[str, np.ndarray],
    score_threshold: float,
    margin_threshold: float,
) -> np.ndarray:
    return (
        (retrieval["top1_scores"] >= score_threshold)
        & (retrieval["top1_margins"] >= margin_threshold)
    )


def _calibrate_gate(
    positive: Mapping[str, np.ndarray],
    locality: Mapping[str, np.ndarray],
    gate: str,
    target: float,
) -> tuple[float, float]:
    maximum_active = _maximum_false_activations(len(locality["top1_scores"]), target)
    if gate == "score":
        return (
            _score_threshold_for_budget(
                locality["top1_scores"],
                np.ones(len(locality["top1_scores"]), dtype=bool),
                maximum_active,
            ),
            0.0,
        )

    margin_candidates = np.concatenate((
        np.asarray([0.0]),
        np.nextafter(np.unique(locality["top1_margins"]), np.inf),
    ))
    best_key = None
    best_thresholds = None
    for margin_threshold in margin_candidates:
        eligible = locality["top1_margins"] >= margin_threshold
        score_threshold = _score_threshold_for_budget(
            locality["top1_scores"],
            eligible,
            maximum_active,
        )
        positive_active = _active_mask(
            positive,
            score_threshold,
            float(margin_threshold),
        )
        locality_active = _active_mask(
            locality,
            score_threshold,
            float(margin_threshold),
        )
        selection_key = (
            int(np.count_nonzero(positive_active & positive["top1_correct"])),
            -int(np.count_nonzero(locality_active)),
            -float(margin_threshold),
            -score_threshold,
        )
        if best_key is None or selection_key > best_key:
            best_key = selection_key
            best_thresholds = (score_threshold, float(margin_threshold))
    if best_thresholds is None:
        raise RuntimeError("No global threshold pair was selected.")
    return best_thresholds


def _retrieval_metrics(retrieval: Mapping[str, np.ndarray]) -> dict[str, float]:
    margins = retrieval["correct_scores"] - retrieval["best_negative_scores"]
    return {
        "recall_at_1": float(np.mean(retrieval["ranks"] <= 1)),
        "recall_at_5": float(np.mean(retrieval["ranks"] <= 5)),
        "recall_at_10": float(np.mean(retrieval["ranks"] <= 10)),
        "mrr": float(np.mean(1.0 / retrieval["ranks"])),
        "correct_margin_mean": float(margins.mean()),
        "top1_margin_mean": float(retrieval["top1_margins"].mean()),
    }


def _gate_metrics(
    positive: Mapping[str, np.ndarray],
    locality: Mapping[str, np.ndarray],
    score_threshold: float,
    margin_threshold: float,
) -> dict[str, float]:
    positive_active = _active_mask(positive, score_threshold, margin_threshold)
    locality_active = _active_mask(locality, score_threshold, margin_threshold)
    top1_correct = positive["top1_correct"]
    return {
        "positive_activation_rate": float(positive_active.mean()),
        "positive_rejection_rate": float((~positive_active).mean()),
        "correct_routed_rate": float(np.mean(positive_active & top1_correct)),
        "incorrect_routed_rate": float(np.mean(positive_active & ~top1_correct)),
        "locality_false_activation_rate": float(locality_active.mean()),
    }


def _aggregate_runs(runs: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    sections = ("retrieval", "gate")
    aggregate = {}
    for section in sections:
        keys = runs[0][section].keys()
        aggregate[section] = {
            key: {
                "mean": float(np.mean([run[section][key] for run in runs])),
                "std": float(np.std([run[section][key] for run in runs], ddof=0)),
            }
            for key in keys
        }
    return aggregate


def _diagnostics(
    positive_queries: Sequence[Mapping[str, Any]],
    locality_queries: Sequence[Mapping[str, Any]],
    positive: Mapping[str, np.ndarray],
    locality: Mapping[str, np.ndarray],
    bank_indices: np.ndarray,
    records: Sequence[Mapping[str, Any]],
    score_threshold: float,
    margin_threshold: float,
    limit: int,
) -> dict[str, list[dict[str, Any]]]:
    if limit <= 0:
        return {"missed_positive": [], "false_locality": []}
    positive_active = _active_mask(positive, score_threshold, margin_threshold)
    locality_active = _active_mask(locality, score_threshold, margin_threshold)
    missed_positions = np.flatnonzero(
        ~(positive_active & positive["top1_correct"])
    )[:limit]
    locality_positions = np.flatnonzero(locality_active)[:limit]

    def retrieved_record(position: int, retrieval: Mapping[str, np.ndarray]):
        bank_position = int(retrieval["top1_positions"][position])
        return records[int(bank_indices[bank_position])]

    return {
        "missed_positive": [
            {
                "prompt": positive_queries[int(position)]["prompt"],
                "expected_source_index": positive_queries[int(position)]["source_index"],
                "retrieved_edit": retrieved_record(int(position), positive)["edit_prompt"],
                "score": float(positive["top1_scores"][position]),
                "margin": float(positive["top1_margins"][position]),
                "active": bool(positive_active[position]),
                "top1_correct": bool(positive["top1_correct"][position]),
            }
            for position in missed_positions
        ],
        "false_locality": [
            {
                "prompt": locality_queries[int(position)]["prompt"],
                "retrieved_edit": retrieved_record(int(position), locality)["edit_prompt"],
                "score": float(locality["top1_scores"][position]),
                "margin": float(locality["top1_margins"][position]),
            }
            for position in locality_positions
        ],
    }


def _split_calibration_queries(
    records: Sequence[Mapping[str, Any]],
    bank_indices: np.ndarray,
    settings: Mapping[str, Any],
    run_index: int,
) -> dict[str, list[dict[str, Any]]]:
    owner_count = min(int(settings["query_owner_count"]), len(bank_indices))
    owner_positions = _shuffle_indices(
        len(bank_indices),
        _derived_seed(int(settings["base_seed"]), "owners", run_index, len(bank_indices)),
    )[:owner_count]
    split = int(round(owner_count * float(settings["calibration_fraction"])))
    split = min(max(split, 1), owner_count - 1)
    calibration_positive = _owner_queries(records, bank_indices, owner_positions[:split])
    validation_positive = _owner_queries(records, bank_indices, owner_positions[split:])

    locality_records = _shuffle_indices(
        len(bank_indices),
        _derived_seed(int(settings["base_seed"]), "locality", run_index, len(bank_indices)),
    )
    locality_split = int(round(
        len(locality_records) * float(settings["calibration_fraction"])
    ))
    locality_split = min(max(locality_split, 1), len(locality_records) - 1)
    calibration_locality = _unique_prompts(
        records,
        bank_indices[locality_records[:locality_split]],
        "locality_prompts",
    )
    calibration_keys = {_prompt_key(item["prompt"]) for item in calibration_locality}
    validation_locality = [
        item
        for item in _unique_prompts(
            records,
            bank_indices[locality_records[locality_split:]],
            "locality_prompts",
        )
        if _prompt_key(item["prompt"]) not in calibration_keys
    ]

    groups = {
        "calibration_positive": calibration_positive,
        "validation_positive": validation_positive,
        "calibration_locality": calibration_locality,
        "validation_locality": validation_locality,
    }
    limits = {
        "calibration_positive": settings["positive_calibration_query_limit"],
        "validation_positive": settings["positive_validation_query_limit"],
        "calibration_locality": settings["locality_calibration_query_limit"],
        "validation_locality": settings["locality_validation_query_limit"],
    }
    for group_name, queries in groups.items():
        positions = _sample_indices(
            len(queries),
            limits[group_name],
            _derived_seed(
                int(settings["base_seed"]),
                group_name,
                run_index,
                len(bank_indices),
            ),
        )
        groups[group_name] = [queries[int(position)] for position in positions]
        if not groups[group_name]:
            raise ValueError(f"No {group_name} queries were selected.")
    return groups


def _cross_dataset_overlap(
    calibration_records: Sequence[Mapping[str, Any]],
    test_records: Sequence[Mapping[str, Any]],
) -> dict[str, int]:
    calibration_edits = {_prompt_key(record["edit_prompt"]) for record in calibration_records}
    test_edits = {_prompt_key(record["edit_prompt"]) for record in test_records}
    calibration_all = set(calibration_edits)
    test_all = set(test_edits)
    for record in calibration_records:
        calibration_all.update(_prompt_key(prompt) for prompt in record["positive_prompts"])
        calibration_all.update(_prompt_key(prompt) for prompt in record["locality_prompts"])
    for record in test_records:
        test_all.update(_prompt_key(prompt) for prompt in record["positive_prompts"])
        test_all.update(_prompt_key(prompt) for prompt in record["locality_prompts"])
    return {
        "exact_edit_prompt_overlap": len(calibration_edits & test_edits),
        "exact_any_prompt_overlap": len(calibration_all & test_all),
        "calibration_unique_prompts": len(calibration_all),
        "test_unique_prompts": len(test_all),
    }


def run_global_experiment(
    *,
    calibration_records: list[dict[str, Any]],
    test_records: list[dict[str, Any]],
    model,
    model_key: str,
    encode_prompts: Callable[..., np.ndarray],
    embedding_device: str,
    settings: dict[str, Any],
    result_header: dict[str, Any],
    output_path: Path,
    model_load_seconds: float,
) -> dict[str, Any]:
    _validate_settings(settings, len(calibration_records), len(test_records))
    bank_sizes = [int(size) for size in settings["bank_sizes"]]
    max_bank_size = bank_sizes[-1]
    ledger = TimingLedger()
    ledger.add("model_load", model_load_seconds)
    experiment_start = time.perf_counter()
    retrieval_device = str(settings["retrieval_device"])

    calibration_retrievals = {
        bank_size: {
            "calibration_positive": [],
            "validation_positive": [],
            "calibration_locality": [],
            "validation_locality": [],
        }
        for bank_size in bank_sizes
    }
    calibration_run_rows = {bank_size: [] for bank_size in bank_sizes}
    for run_index in range(1, int(settings["n_runs"]) + 1):
        run_seed = int(settings["base_seed"]) + run_index - 1
        order = _shuffle_indices(len(calibration_records), run_seed)
        selected = order[:max_bank_size]
        edit_embeddings = _timed_encode(
            encode_prompts,
            model,
            model_key,
            [calibration_records[int(index)]["edit_prompt"] for index in selected],
            "passage",
            ledger,
            "calibration_edit_encoding",
            embedding_device,
        )
        for bank_size in bank_sizes:
            bank_indices = selected[:bank_size]
            groups = _split_calibration_queries(
                calibration_records,
                bank_indices,
                settings,
                run_index,
            )
            query_names = tuple(groups)
            query_prompts = [
                query["prompt"]
                for group_name in query_names
                for query in groups[group_name]
            ]
            query_embeddings = _timed_encode(
                encode_prompts,
                model,
                model_key,
                query_prompts,
                "query",
                ledger,
                "calibration_query_encoding",
                embedding_device,
            )
            stops = np.cumsum([len(groups[name]) for name in query_names])
            chunks = dict(zip(
                query_names,
                np.split(query_embeddings, stops[:-1]),
                strict=True,
            ))

            retrieval_start = time.perf_counter()
            for group_name in ("calibration_positive", "validation_positive"):
                owners = np.asarray([
                    query["owner_position"] for query in groups[group_name]
                ])
                retrieval = _positive_retrieval(
                    chunks[group_name],
                    edit_embeddings[:bank_size],
                    owners,
                    int(settings["retrieval_batch_size"]),
                    retrieval_device,
                )
                calibration_retrievals[bank_size][group_name].append(retrieval)
            for group_name in ("calibration_locality", "validation_locality"):
                retrieval = _locality_retrieval(
                    chunks[group_name],
                    edit_embeddings[:bank_size],
                    int(settings["retrieval_batch_size"]),
                    retrieval_device,
                )
                calibration_retrievals[bank_size][group_name].append(retrieval)
            _sync_device(retrieval_device)
            ledger.add("calibration_retrieval", time.perf_counter() - retrieval_start)
            calibration_run_rows[bank_size].append({
                "run": run_index,
                "seed": run_seed,
                "query_counts": {name: len(groups[name]) for name in query_names},
            })

    thresholds = {}
    calibration_summary = {}
    threshold_start = time.perf_counter()
    for bank_size in bank_sizes:
        pooled = {
            name: _concatenate(values)
            for name, values in calibration_retrievals[bank_size].items()
        }
        score_threshold, margin_threshold = _calibrate_gate(
            pooled["calibration_positive"],
            pooled["calibration_locality"],
            str(settings["gate"]),
            float(settings["locality_far_target"]),
        )
        thresholds[bank_size] = (score_threshold, margin_threshold)
        calibration_summary[bank_size] = {
            "score_threshold": score_threshold,
            "margin_threshold": margin_threshold,
            "calibration": {
                **_retrieval_metrics(pooled["calibration_positive"]),
                **_gate_metrics(
                    pooled["calibration_positive"],
                    pooled["calibration_locality"],
                    score_threshold,
                    margin_threshold,
                ),
            },
            "validation": {
                **_retrieval_metrics(pooled["validation_positive"]),
                **_gate_metrics(
                    pooled["validation_positive"],
                    pooled["validation_locality"],
                    score_threshold,
                    margin_threshold,
                ),
            },
            "runs": calibration_run_rows[bank_size],
        }
    ledger.add("threshold_selection", time.perf_counter() - threshold_start)

    test_runs_by_size = {bank_size: [] for bank_size in bank_sizes}
    for run_index in range(1, int(settings["n_runs"]) + 1):
        run_seed = int(settings["base_seed"]) + run_index - 1
        order = _shuffle_indices(len(test_records), run_seed)
        selected = order[:max_bank_size]
        edit_embeddings = _timed_encode(
            encode_prompts,
            model,
            model_key,
            [test_records[int(index)]["edit_prompt"] for index in selected],
            "passage",
            ledger,
            "test_edit_encoding",
            embedding_device,
        )
        for bank_size in bank_sizes:
            bank_indices = selected[:bank_size]
            owner_positions = np.arange(bank_size, dtype=np.int64)
            positive_queries = _owner_queries(
                test_records,
                bank_indices,
                owner_positions,
            )
            locality_queries = _unique_prompts(
                test_records,
                bank_indices,
                "locality_prompts",
            )
            positive_positions = _sample_indices(
                len(positive_queries),
                settings["test_positive_query_limit"],
                _derived_seed(run_seed, "test_positive", bank_size),
            )
            locality_positions = _sample_indices(
                len(locality_queries),
                settings["test_locality_query_limit"],
                _derived_seed(run_seed, "test_locality", bank_size),
            )
            positive_queries = [positive_queries[int(index)] for index in positive_positions]
            locality_queries = [locality_queries[int(index)] for index in locality_positions]
            all_queries = [
                query["prompt"] for query in (*positive_queries, *locality_queries)
            ]
            query_embeddings = _timed_encode(
                encode_prompts,
                model,
                model_key,
                all_queries,
                "query",
                ledger,
                "test_query_encoding",
                embedding_device,
            )
            positive_embeddings = query_embeddings[:len(positive_queries)]
            locality_embeddings = query_embeddings[len(positive_queries):]
            owners = np.asarray([
                query["owner_position"] for query in positive_queries
            ])

            retrieval_start = time.perf_counter()
            positive = _positive_retrieval(
                positive_embeddings,
                edit_embeddings[:bank_size],
                owners,
                int(settings["retrieval_batch_size"]),
                retrieval_device,
            )
            locality = _locality_retrieval(
                locality_embeddings,
                edit_embeddings[:bank_size],
                int(settings["retrieval_batch_size"]),
                retrieval_device,
            )
            _sync_device(retrieval_device)
            ledger.add("test_retrieval", time.perf_counter() - retrieval_start)
            score_threshold, margin_threshold = thresholds[bank_size]
            test_runs_by_size[bank_size].append({
                "run": run_index,
                "seed": run_seed,
                "positive_query_count": len(positive_queries),
                "locality_query_count": len(locality_queries),
                "retrieval": _retrieval_metrics(positive),
                "gate": _gate_metrics(
                    positive,
                    locality,
                    score_threshold,
                    margin_threshold,
                ),
                "diagnostics": _diagnostics(
                    positive_queries,
                    locality_queries,
                    positive,
                    locality,
                    bank_indices,
                    test_records,
                    score_threshold,
                    margin_threshold,
                    int(settings["diagnostic_limit"]),
                ),
            })

    total_seconds = time.perf_counter() - experiment_start
    ledger.add("experiment_total_excluding_model_load", total_seconds)
    ledger.add("experiment_total_including_model_load", total_seconds + model_load_seconds)
    cuda_memory = None
    if embedding_device.startswith("cuda") and torch.cuda.is_available():
        device = torch.device(embedding_device)
        cuda_memory = {
            "peak_allocated_gb": round(torch.cuda.max_memory_allocated(device) / 1024**3, 6),
            "peak_reserved_gb": round(torch.cuda.max_memory_reserved(device) / 1024**3, 6),
        }

    scales = []
    for bank_size in bank_sizes:
        runs = test_runs_by_size[bank_size]
        scales.append({
            "bank_size": bank_size,
            "thresholds": {
                "score": thresholds[bank_size][0],
                "margin": thresholds[bank_size][1],
            },
            "calibration": calibration_summary[bank_size],
            "test_runs": runs,
            "test_aggregate": _aggregate_runs(runs),
        })

    result = dict(result_header)
    result.update({
        "protocol": {
            "calibration_dataset": "uniedit/similarity",
            "test_dataset": "counterfact/final",
            "threshold_policy": "pooled_uniedit_calibration_frozen_for_counterfact",
            "run_sampling": "Random(base_seed + run - 1).shuffle",
            "timing_scope": (
                "model loading, prompt encoding, retrieval, and threshold selection; "
                "dataset loading and JSON serialization are excluded"
            ),
            "settings": settings,
        },
        "datasets": {
            "calibration": dataset_identity(calibration_records),
            "test": dataset_identity(test_records),
            "overlap": _cross_dataset_overlap(calibration_records, test_records),
        },
        "timing": ledger.as_dict(),
        "cuda_memory": cuda_memory,
        "scales": scales,
    })
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_suffix(f"{output_path.suffix}.tmp")
    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(result, file, indent=2, ensure_ascii=True, allow_nan=False)
    temporary.replace(output_path)
    return result


def global_result_tables(results: Sequence[tuple[str, Mapping[str, Any]]]) -> dict[str, pd.DataFrame]:
    accuracy_rows = []
    timing_rows = []
    for model_key, result in results:
        for scale in result["scales"]:
            aggregate = scale["test_aggregate"]
            accuracy_rows.append({
                "Embedding model": model_key,
                "Bank size": scale["bank_size"],
                "Score threshold": scale["thresholds"]["score"],
                "Margin threshold": scale["thresholds"]["margin"],
                "Recall@1": aggregate["retrieval"]["recall_at_1"]["mean"],
                "MRR": aggregate["retrieval"]["mrr"]["mean"],
                "Correct routed": aggregate["gate"]["correct_routed_rate"]["mean"],
                "Incorrect routed": aggregate["gate"]["incorrect_routed_rate"]["mean"],
                "Positive rejected": aggregate["gate"]["positive_rejection_rate"]["mean"],
                "Locality FAR": aggregate["gate"]["locality_false_activation_rate"]["mean"],
            })
        timing = result["timing"]
        seconds = timing["seconds"]
        rates = timing["items_per_second"]
        memory = result.get("cuda_memory") or {}
        timing_rows.append({
            "Embedding model": model_key,
            "Load seconds": seconds.get("model_load"),
            "Total seconds": seconds.get("experiment_total_including_model_load"),
            "Calibration edit prompts/s": rates.get("calibration_edit_encoding"),
            "Calibration query prompts/s": rates.get("calibration_query_encoding"),
            "Test edit prompts/s": rates.get("test_edit_encoding"),
            "Test query prompts/s": rates.get("test_query_encoding"),
            "Peak allocated GiB": memory.get("peak_allocated_gb"),
            "Peak reserved GiB": memory.get("peak_reserved_gb"),
        })
    return {
        "accuracy": pd.DataFrame(accuracy_rows).set_index([
            "Embedding model",
            "Bank size",
        ]),
        "timing": pd.DataFrame(timing_rows).set_index(
            "Embedding model"
        ),
    }

In [ ]:
def build_result_header(
    model,
    model_key: str,
    dataset_metadata: dict[str, Any],
    experiment: dict[str, Any],
) -> dict[str, Any]:
    spec = EMBEDDING_MODEL_SPECS[model_key]
    return {
        "embedding_model_key": model_key,
        "embedding_model": spec["repo"],
        "embedding_backend": spec["backend"],
        "input_prefixes": spec["prefixes"],
        "representation": REPRESENTATION_TYPE,
        "use_fp16": EMBEDDING_USE_FP16,
        "compute_dtype": embedding_compute_dtype(),
        "max_length_cap": EMBEDDING_MAX_LENGTH,
        "effective_max_length": embedding_effective_max_length(model, model_key),
        "dataset": dataset_metadata,
        "experiment": experiment,
    }


def experiment_configuration(
    experiment_mode: str,
    record_limit: int | None,
    record_batch_size: int,
    global_bank_sizes: list[int],
    global_gate: str,
    global_locality_far_target: float,
    global_n_runs: int,
    global_base_seed: int,
    global_query_owner_count: int,
    global_calibration_fraction: float,
    global_positive_calibration_query_limit: int | None,
    global_positive_validation_query_limit: int | None,
    global_locality_calibration_query_limit: int | None,
    global_locality_validation_query_limit: int | None,
    global_test_positive_query_limit: int | None,
    global_test_locality_query_limit: int | None,
    global_diagnostic_limit: int,
    global_retrieval_batch_size: int,
    global_retrieval_device: str,
) -> dict[str, Any]:
    if experiment_mode not in {"local", "global"}:
        raise ValueError("experiment_mode must be 'local' or 'global'.")
    if record_limit is not None and int(record_limit) <= 0:
        raise ValueError("record_limit must be positive when set.")
    if record_batch_size <= 0:
        raise ValueError("record_batch_size must be positive.")
    if experiment_mode == "local":
        return {
            "mode": "local",
            "record_limit": record_limit,
            "settings": {"record_batch_size": int(record_batch_size)},
        }

    bank_sizes = [int(size) for size in global_bank_sizes]
    if not bank_sizes or any(size < 2 for size in bank_sizes):
        raise ValueError("global_bank_sizes must contain integers above one.")
    if bank_sizes != sorted(set(bank_sizes)):
        raise ValueError("global_bank_sizes must be unique and increasing.")
    if global_gate not in {"score", "score_margin"}:
        raise ValueError("global_gate must be 'score' or 'score_margin'.")
    if not 0.0 < float(global_locality_far_target) < 1.0:
        raise ValueError("global_locality_far_target must be in (0, 1).")
    if int(global_n_runs) <= 0:
        raise ValueError("global_n_runs must be positive.")
    if not 2 <= int(global_query_owner_count) <= bank_sizes[0]:
        raise ValueError(
            "global_query_owner_count must be between two and the smallest bank size."
        )
    if not 0.0 < float(global_calibration_fraction) < 1.0:
        raise ValueError("global_calibration_fraction must be in (0, 1).")
    query_limits = (
        global_positive_calibration_query_limit,
        global_positive_validation_query_limit,
        global_locality_calibration_query_limit,
        global_locality_validation_query_limit,
        global_test_positive_query_limit,
        global_test_locality_query_limit,
    )
    if any(limit is not None and int(limit) <= 0 for limit in query_limits):
        raise ValueError("Global query limits must be positive when set.")
    if int(global_diagnostic_limit) < 0:
        raise ValueError("global_diagnostic_limit cannot be negative.")
    if int(global_retrieval_batch_size) <= 0:
        raise ValueError("global_retrieval_batch_size must be positive.")
    if (
        global_retrieval_device != "cpu"
        and not str(global_retrieval_device).startswith("cuda")
    ):
        raise ValueError("global_retrieval_device must be 'cpu' or a CUDA device.")

    return {
        "mode": "global",
        "record_limit": None,
        "settings": {
            "bank_sizes": bank_sizes,
            "gate": global_gate,
            "locality_far_target": float(global_locality_far_target),
            "n_runs": int(global_n_runs),
            "base_seed": int(global_base_seed),
            "query_owner_count": int(global_query_owner_count),
            "calibration_fraction": float(global_calibration_fraction),
            "positive_calibration_query_limit": global_positive_calibration_query_limit,
            "positive_validation_query_limit": global_positive_validation_query_limit,
            "locality_calibration_query_limit": global_locality_calibration_query_limit,
            "locality_validation_query_limit": global_locality_validation_query_limit,
            "test_positive_query_limit": global_test_positive_query_limit,
            "test_locality_query_limit": global_test_locality_query_limit,
            "diagnostic_limit": int(global_diagnostic_limit),
            "retrieval_batch_size": int(global_retrieval_batch_size),
            "retrieval_device": global_retrieval_device,
        },
    }


def similarity_dataset_metadata_matches(
    saved: dict[str, Any] | None,
    current: dict[str, Any],
) -> bool:
    if saved is None:
        return False
    if saved.get("prompt_fingerprint") and current.get("prompt_fingerprint"):
        identity_fields = (
            "source",
            "rows",
            "prompt_fingerprint",
            "dataset_id",
            "repo",
            "config",
            "split",
        )
        return all(saved.get(field) == current.get(field) for field in identity_fields)
    return saved == current


def local_result_matches_configuration(
    output_path: Path,
    model_key: str,
    experiment: dict[str, Any],
    dataset_metadata: dict[str, Any],
) -> bool:
    if not output_path.exists():
        return False
    with open(output_path, encoding="utf-8") as file:
        result = json.load(file)
    spec = EMBEDDING_MODEL_SPECS[model_key]
    return (
        result.get("embedding_model_key") == model_key
        and result.get("embedding_model") == spec["repo"]
        and result.get("embedding_backend") == spec["backend"]
        and result.get("input_prefixes") == spec["prefixes"]
        and result.get("representation") == REPRESENTATION_TYPE
        and result.get("use_fp16") == EMBEDDING_USE_FP16
        and result.get("compute_dtype") == embedding_compute_dtype()
        and result.get("max_length_cap") == EMBEDDING_MAX_LENGTH
        and result.get("experiment") == experiment
        and similarity_dataset_metadata_matches(
            result.get("dataset"),
            dataset_metadata,
        )
    )


def calculate_local_similarity(
    embedding_model_keys: list[str],
    dataset_source: str,
    record_limit: int | None,
    record_batch_size: int,
    overwrite_output: bool,
    experiment: dict[str, Any],
) -> None:
    model_keys = resolve_embedding_model_keys(embedding_model_keys)
    dataset = load_similarity_dataset(dataset_source)
    dataset_metadata = similarity_dataset_metadata(dataset, dataset_source)
    records = build_local_similarity_records(dataset, record_limit)
    failures = []
    for model_key in model_keys:
        output_path = local_output_path(model_key, record_limit)
        if output_path.exists() and not overwrite_output:
            if local_result_matches_configuration(
                output_path,
                model_key,
                experiment,
                dataset_metadata,
            ):
                print(f"skipped_existing={output_path}")
                continue
            print(f"recalculating_stale={output_path}")

        model = None
        try:
            model = load_embedding_model(model_key)
            run_local_similarity_analysis(
                dataset,
                records,
                model=model,
                model_key=model_key,
                output_path=output_path,
                record_batch_size=record_batch_size,
                dataset_metadata=dataset_metadata,
                experiment=experiment,
            )
        except Exception as error:
            failure = f"{model_key}: {type(error).__name__}: {error}"
            failures.append(failure)
            print(f"embedding_model_failed={failure}")
        finally:
            if model is not None:
                move_embedding_model_to_cpu(model, model_key)
            model = None
            clear_embedding_memory(model_key)

    dataset = None
    records = None
    gc.collect()
    if failures:
        raise RuntimeError("Embedding models failed:\n" + "\n".join(failures))


def load_global_datasets(dataset_source: str):
    if dataset_source == "local":
        calibration = load_from_disk(str(LOCAL_DATASET_PATH))["similarity"]
        test = load_from_disk(str(LOCAL_COUNTERFACT_PATH))
        return calibration, test
    if dataset_source == "huggingface":
        calibration = load_dataset(
            DATASET_REPO,
            DATASET_CONFIG,
            split=DATASET_SPLIT,
        )
        test = load_dataset(
            DATASET_REPO,
            COUNTERFACT_DATASET_CONFIG,
            split=DATASET_SPLIT,
        )
        return calibration, test
    raise ValueError("dataset_source must be 'local' or 'huggingface'.")


def global_output_path(model_key: str) -> Path:
    return OUTPUT_DIR / "global" / embedding_precision_mode() / f"{model_key}.json"


def calculate_global_similarity(
    embedding_model_keys: list[str],
    dataset_source: str,
    overwrite_output: bool,
    settings: dict[str, Any],
) -> None:
    model_keys = resolve_embedding_model_keys(embedding_model_keys)
    calibration_dataset, test_dataset = load_global_datasets(dataset_source)
    calibration_records = build_uniedit_records(calibration_dataset)
    test_records = build_counterfact_records(test_dataset)
    calibration_identity = dataset_identity(calibration_records)
    test_identity = dataset_identity(test_records)
    dataset_header = {
        "source": dataset_source,
        "calibration": "uniedit/similarity",
        "test": "counterfact/final",
    }
    experiment = {"mode": "global", "settings": settings}
    failures = []
    for model_key in model_keys:
        output_path = global_output_path(model_key)
        if output_path.exists() and not overwrite_output:
            with open(output_path, encoding="utf-8") as file:
                existing = json.load(file)
            if (
                existing.get("embedding_model_key") == model_key
                and existing.get("protocol", {}).get("settings") == settings
                and existing.get("compute_dtype") == embedding_compute_dtype()
                and existing.get("datasets", {}).get("calibration")
                == calibration_identity
                and existing.get("datasets", {}).get("test") == test_identity
            ):
                print(f"skipped_existing={output_path}")
                continue
            print(f"recalculating_stale={output_path}")

        model = None
        try:
            device = embedding_device()
            if device.startswith("cuda") and torch.cuda.is_available():
                torch.cuda.synchronize(torch.device(device))
                torch.cuda.reset_peak_memory_stats(torch.device(device))
            load_start = perf_counter()
            model = load_embedding_model(model_key)
            if device.startswith("cuda") and torch.cuda.is_available():
                torch.cuda.synchronize(torch.device(device))
            model_load_seconds = perf_counter() - load_start
            result_header = build_result_header(
                model,
                model_key,
                dataset_header,
                experiment,
            )
            result = run_global_experiment(
                calibration_records=calibration_records,
                test_records=test_records,
                model=model,
                model_key=model_key,
                encode_prompts=encode_prompts,
                embedding_device=device,
                settings=settings,
                result_header=result_header,
                output_path=output_path,
                model_load_seconds=model_load_seconds,
            )
            total = result["timing"]["seconds"][
                "experiment_total_including_model_load"
            ]
            print(f"saved={output_path} total_seconds={total:.2f}")
        except Exception as error:
            failure = f"{model_key}: {type(error).__name__}: {error}"
            failures.append(failure)
            print(f"embedding_model_failed={failure}")
        finally:
            if model is not None:
                move_embedding_model_to_cpu(model, model_key)
            model = None
            clear_embedding_memory(model_key)

    calibration_dataset = None
    test_dataset = None
    calibration_records = None
    test_records = None
    gc.collect()
    if failures:
        raise RuntimeError("Embedding models failed:\n" + "\n".join(failures))


def load_global_results(
    embedding_model_keys: list[str],
    settings: dict[str, Any],
) -> list[tuple[str, dict]]:
    results = []
    for model_key in resolve_embedding_model_keys(embedding_model_keys):
        output_path = global_output_path(model_key)
        if not output_path.exists():
            raise FileNotFoundError(f"Missing global result: {output_path}")
        with open(output_path, encoding="utf-8") as file:
            result = json.load(file)
        if (
            result.get("embedding_model_key") != model_key
            or result.get("protocol", {}).get("settings") != settings
            or result.get("compute_dtype") != embedding_compute_dtype()
        ):
            raise ValueError(f"Stale global result: {output_path}. Recalculate it.")
        results.append((model_key, result))
    return results


def calculate_similarity() -> None:
    configuration = experiment_configuration(
        experiment_mode=EXPERIMENT_MODE,
        record_limit=RECORD_LIMIT,
        record_batch_size=RECORD_BATCH_SIZE,
        global_bank_sizes=GLOBAL_BANK_SIZES,
        global_gate=GLOBAL_GATE,
        global_locality_far_target=GLOBAL_LOCALITY_FAR_TARGET,
        global_n_runs=GLOBAL_N_RUNS,
        global_base_seed=GLOBAL_BASE_SEED,
        global_query_owner_count=GLOBAL_QUERY_OWNER_COUNT,
        global_calibration_fraction=GLOBAL_CALIBRATION_FRACTION,
        global_positive_calibration_query_limit=(
            GLOBAL_POSITIVE_CALIBRATION_QUERY_LIMIT
        ),
        global_positive_validation_query_limit=(
            GLOBAL_POSITIVE_VALIDATION_QUERY_LIMIT
        ),
        global_locality_calibration_query_limit=(
            GLOBAL_LOCALITY_CALIBRATION_QUERY_LIMIT
        ),
        global_locality_validation_query_limit=(
            GLOBAL_LOCALITY_VALIDATION_QUERY_LIMIT
        ),
        global_test_positive_query_limit=GLOBAL_TEST_POSITIVE_QUERY_LIMIT,
        global_test_locality_query_limit=GLOBAL_TEST_LOCALITY_QUERY_LIMIT,
        global_diagnostic_limit=GLOBAL_DIAGNOSTIC_LIMIT,
        global_retrieval_batch_size=GLOBAL_RETRIEVAL_BATCH_SIZE,
        global_retrieval_device=GLOBAL_RETRIEVAL_DEVICE,
    )
    if configuration["mode"] == "local":
        calculate_local_similarity(
            embedding_model_keys=EMBEDDING_MODEL_KEYS,
            dataset_source=DATASET_SOURCE,
            record_limit=RECORD_LIMIT,
            record_batch_size=RECORD_BATCH_SIZE,
            overwrite_output=OVERWRITE_EXISTING_RESULT,
            experiment=configuration,
        )
        return
    calculate_global_similarity(
        embedding_model_keys=EMBEDDING_MODEL_KEYS,
        dataset_source=DATASET_SOURCE,
        overwrite_output=OVERWRITE_EXISTING_RESULT,
        settings=configuration["settings"],
    )


def visualize_similarity() -> None:
    configuration = experiment_configuration(
        experiment_mode=EXPERIMENT_MODE,
        record_limit=RECORD_LIMIT,
        record_batch_size=RECORD_BATCH_SIZE,
        global_bank_sizes=GLOBAL_BANK_SIZES,
        global_gate=GLOBAL_GATE,
        global_locality_far_target=GLOBAL_LOCALITY_FAR_TARGET,
        global_n_runs=GLOBAL_N_RUNS,
        global_base_seed=GLOBAL_BASE_SEED,
        global_query_owner_count=GLOBAL_QUERY_OWNER_COUNT,
        global_calibration_fraction=GLOBAL_CALIBRATION_FRACTION,
        global_positive_calibration_query_limit=(
            GLOBAL_POSITIVE_CALIBRATION_QUERY_LIMIT
        ),
        global_positive_validation_query_limit=(
            GLOBAL_POSITIVE_VALIDATION_QUERY_LIMIT
        ),
        global_locality_calibration_query_limit=(
            GLOBAL_LOCALITY_CALIBRATION_QUERY_LIMIT
        ),
        global_locality_validation_query_limit=(
            GLOBAL_LOCALITY_VALIDATION_QUERY_LIMIT
        ),
        global_test_positive_query_limit=GLOBAL_TEST_POSITIVE_QUERY_LIMIT,
        global_test_locality_query_limit=GLOBAL_TEST_LOCALITY_QUERY_LIMIT,
        global_diagnostic_limit=GLOBAL_DIAGNOSTIC_LIMIT,
        global_retrieval_batch_size=GLOBAL_RETRIEVAL_BATCH_SIZE,
        global_retrieval_device=GLOBAL_RETRIEVAL_DEVICE,
    )
    if configuration["mode"] == "local":
        visualize_local_similarity(
            embedding_model_keys=EMBEDDING_MODEL_KEYS,
            dataset_source=DATASET_SOURCE,
            record_limit=RECORD_LIMIT,
            show_std=PLOT_SHOW_STD,
            experiment=configuration,
        )
        return
    visualize_global_similarity(
        EMBEDDING_MODEL_KEYS,
        configuration["settings"],
    )

## Tables And Plots

In [ ]:
PAIR_COLORS = {
    "p1-p2": "#0072B2",
    "p1-p3": "#E69F00",
    "p2-p3": "#009E73",
}


def aggregate_pair_table(results: list[tuple[str, dict]], show_std: bool) -> pd.DataFrame:
    rows = []
    for model_key, result in results:
        row = {"Embedding model": model_key}
        for pair_key, _, _, label in PAIR_SPECS:
            pair = result["aggregate"]["pairs"][pair_key]
            row[label] = (
                f"{pair['mean']:.4f} +/- {pair['std']:.4f}"
                if show_std
                else f"{pair['mean']:.4f}"
            )
        rows.append(row)
    return pd.DataFrame(rows).set_index("Embedding model")


def observation_count_table(results: list[tuple[str, dict]]) -> pd.DataFrame:
    rows = []
    for model_key, result in results:
        row = {
            "Embedding model": model_key,
            "Records": result["aggregate"]["record_count"],
        }
        for pair_key, _, _, label in PAIR_SPECS:
            row[label] = result["aggregate"]["pairs"][pair_key]["count"]
        rows.append(row)
    return pd.DataFrame(rows).set_index("Embedding model")


def summarize_pairwise_preference(result: dict) -> dict:
    tolerance = 1e-12
    gaps = np.concatenate([
        (
            np.asarray(record["pair_values"]["p1-p2"], dtype=np.float64)[:, None]
            - np.asarray(record["pair_values"]["p1-p3"], dtype=np.float64)[None, :]
        ).reshape(-1)
        for record in result["records"]
    ])
    generality_preferred = int(np.count_nonzero(gaps > tolerance))
    locality_preferred = int(np.count_nonzero(gaps < -tolerance))
    ties = int(gaps.size - generality_preferred - locality_preferred)
    return {
        "comparisons": int(gaps.size),
        "mean_gap": float(gaps.mean()),
        "gap_std": float(gaps.std(ddof=0)),
        "generality_preferred": generality_preferred / gaps.size,
        "locality_preferred": locality_preferred / gaps.size,
        "ties": ties / gaps.size,
    }


def pairwise_preference_table(results: list[tuple[str, dict]]) -> pd.DataFrame:
    rows = []
    for model_key, result in results:
        preference = result["pairwise_preference"]
        rows.append({
            "Embedding model": model_key,
            "Comparisons": preference["comparisons"],
            "Mean gap": preference["mean_gap"],
            "Gap std": preference["gap_std"],
            "Generality preferred": preference["generality_preferred"],
            "Locality preferred": preference["locality_preferred"],
            "Ties": preference["ties"],
        })
    return pd.DataFrame(rows).set_index("Embedding model")


def plot_local_similarity(results: list[tuple[str, dict]], show_std: bool) -> None:
    model_keys = [model_key for model_key, _ in results]
    pair_specs = list(PAIR_SPECS)
    positions = np.arange(len(model_keys), dtype=float)
    group_width = 0.78
    bar_width = group_width / len(pair_specs)
    offsets = (
        np.arange(len(pair_specs), dtype=float) - (len(pair_specs) - 1) / 2
    ) * bar_width

    all_lower = []
    all_upper = []
    figure, axis = plt.subplots(figsize=(18, 7))
    for pair_offset, (pair_key, _, _, label) in zip(offsets, pair_specs):
        means = np.asarray([
            result["aggregate"]["pairs"][pair_key]["mean"]
            for _, result in results
        ])
        stds = np.asarray([
            result["aggregate"]["pairs"][pair_key]["std"]
            for _, result in results
        ])
        axis.bar(
            positions + pair_offset,
            means,
            width=bar_width * 0.92,
            yerr=stds if show_std else None,
            label=f"{pair_key}: {label}",
            color=PAIR_COLORS[pair_key],
            capsize=5 if show_std else 0,
            error_kw={"elinewidth": 1.25, "capthick": 1.25},
        )
        all_lower.extend((means - stds if show_std else means).tolist())
        all_upper.extend((means + stds if show_std else means).tolist())

    y_min = min(0.0, min(all_lower))
    y_max = max(all_upper)
    padding = max(0.05, (y_max - y_min) * 0.12)
    axis.set_ylim(
        max(-1.0, y_min - (padding if y_min < 0 else 0)),
        min(1.0, y_max + padding),
    )
    axis.yaxis.set_major_locator(MaxNLocator(nbins=10))
    axis.set_xticks(positions, model_keys, rotation=20, ha="right")
    axis.set_ylabel("cosine similarity")
    record_counts = {result["aggregate"]["record_count"] for _, result in results}
    count_label = str(record_counts.pop()) if len(record_counts) == 1 else "mixed"
    axis.set_title(f"Local prompt similarity | n={count_label}", fontweight="bold")
    axis.grid(axis="y", alpha=0.22)
    axis.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), title="prompt pair")
    figure.tight_layout(rect=(0, 0, 0.82, 1))
    plt.show()


def visualize_local_similarity(
    embedding_model_keys: list[str],
    dataset_source: str,
    record_limit: int | None,
    show_std: bool,
    experiment: dict[str, Any],
) -> None:
    dataset = load_similarity_dataset(dataset_source)
    dataset_metadata = similarity_dataset_metadata(dataset, dataset_source)
    results = []
    for model_key in resolve_embedding_model_keys(embedding_model_keys):
        output_path = local_output_path(model_key, record_limit)
        if not output_path.exists():
            raise FileNotFoundError(f"Missing local result: {output_path}")
        if not local_result_matches_configuration(
            output_path,
            model_key,
            experiment,
            dataset_metadata,
        ):
            raise ValueError(f"Stale local result: {output_path}. Recalculate it.")
        with open(output_path, encoding="utf-8") as file:
            result = json.load(file)
        result["pairwise_preference"] = summarize_pairwise_preference(result)
        result.pop("records", None)
        results.append((model_key, result))
    dataset = None
    gc.collect()

    display(aggregate_pair_table(results, show_std=show_std).style)
    display(pairwise_preference_table(results).style.format({
        "Comparisons": "{:,}",
        "Mean gap": "{:.4f}",
        "Gap std": "{:.4f}",
        "Generality preferred": "{:.4f}",
        "Locality preferred": "{:.4f}",
        "Ties": "{:.4f}",
    }))
    display(observation_count_table(results).style.format("{:,}"))
    plot_local_similarity(results, show_std=show_std)


def global_protocol_table() -> pd.DataFrame:
    return pd.DataFrame([
        {
            "Stage": "1. Tune",
            "Dataset": "UniEdit calibration split",
            "Purpose": "Choose score and margin thresholds at the target locality FAR.",
        },
        {
            "Stage": "2. Validate",
            "Dataset": "Held-out UniEdit split",
            "Purpose": "Check that calibration works beyond the prompts used to choose thresholds.",
        },
        {
            "Stage": "3. Freeze",
            "Dataset": "No data",
            "Purpose": "Lock each bank-size threshold before CounterFact is inspected.",
        },
        {
            "Stage": "4. Test",
            "Dataset": "Seeded CounterFact banks",
            "Purpose": "Measure cross-dataset retrieval, routing, rejection, and false activation.",
        },
        {
            "Stage": "5. Compare",
            "Dataset": "All results",
            "Purpose": "Trade routing quality against encoder time, throughput, and CUDA memory.",
        },
    ]).set_index("Stage")


def global_calibration_table(results: list[tuple[str, dict]]) -> pd.DataFrame:
    rows = []
    for model_key, result in results:
        for scale in result["scales"]:
            validation = scale["calibration"]["validation"]
            rows.append({
                "Embedding model": model_key,
                "Bank size": scale["bank_size"],
                "Score threshold": scale["thresholds"]["score"],
                "Margin threshold": scale["thresholds"]["margin"],
                "UniEdit Recall@1": validation["recall_at_1"],
                "UniEdit correct routed": validation["correct_routed_rate"],
                "UniEdit locality FAR": validation["locality_false_activation_rate"],
            })
    return pd.DataFrame(rows).set_index([
        "Embedding model",
        "Bank size",
    ])


def global_candidate_table(results: list[tuple[str, dict]]) -> pd.DataFrame:
    largest_bank = max(
        scale["bank_size"]
        for _, result in results
        for scale in result["scales"]
    )
    rows = []
    for model_key, result in results:
        scale = next(
            item for item in result["scales"]
            if item["bank_size"] == largest_bank
        )
        aggregate = scale["test_aggregate"]
        timing = result["timing"]
        memory = result.get("cuda_memory") or {}
        rows.append({
            "Embedding model": model_key,
            "Recall@1": aggregate["retrieval"]["recall_at_1"]["mean"],
            "Correct routed": aggregate["gate"]["correct_routed_rate"]["mean"],
            "Incorrect routed": aggregate["gate"]["incorrect_routed_rate"]["mean"],
            "Positive rejected": aggregate["gate"]["positive_rejection_rate"]["mean"],
            "Locality FAR": aggregate["gate"]["locality_false_activation_rate"]["mean"],
            "Query prompts/s": timing["items_per_second"].get("test_query_encoding"),
            "Total seconds": timing["seconds"].get("experiment_total_including_model_load"),
            "Peak allocated GiB": memory.get("peak_allocated_gb"),
        })
    return pd.DataFrame(rows).set_index("Embedding model").sort_values(
        ["Correct routed", "Recall@1"],
        ascending=False,
    )


def plot_global_quality_speed(results: list[tuple[str, dict]]) -> None:
    largest_bank = max(
        scale["bank_size"]
        for _, result in results
        for scale in result["scales"]
    )
    figure, axis = plt.subplots(figsize=(9, 6))
    for model_key, result in results:
        scale = next(
            item for item in result["scales"]
            if item["bank_size"] == largest_bank
        )
        throughput = result["timing"]["items_per_second"].get(
            "test_query_encoding",
            0.0,
        )
        correct = scale["test_aggregate"]["gate"][
            "correct_routed_rate"
        ]["mean"]
        axis.scatter(throughput, correct, s=90)
        axis.annotate(
            model_key,
            (throughput, correct),
            xytext=(7, 7),
            textcoords="offset points",
        )
    axis.set_xlabel("CounterFact query encoding throughput (prompts/s)")
    axis.set_ylabel("CounterFact correct routed rate")
    axis.set_title(
        f"Cross-dataset quality-speed tradeoff | bank={largest_bank}",
        fontweight="bold",
    )
    axis.grid(alpha=0.25)
    figure.tight_layout()
    plt.show()


def visualize_global_similarity(
    embedding_model_keys: list[str],
    settings: dict[str, Any],
) -> None:
    results = load_global_results(embedding_model_keys, settings)
    tables = global_result_tables(results)
    display(global_protocol_table().style)
    display(global_calibration_table(results).style)
    display(global_candidate_table(results).style)
    display(tables["accuracy"].style)
    display(tables["timing"].style)
    overlap = pd.DataFrame([
        {
            "Embedding model": model_key,
            **result["datasets"]["overlap"],
        }
        for model_key, result in results
    ]).set_index("Embedding model")
    display(overlap.style)
    plot_global_quality_speed(results)

## Experiment Configuration And Actions

Choose one mode, review its validated configuration, then run the commented calculation and visualization actions in the following cell.

In [ ]:
# Select within-record diagnostics or cross-dataset global routing.
# Options: "local", "global".
EXPERIMENT_MODE = "global"

# Load datasets from Hugging Face or local Arrow artifacts.
# Options: "local", "huggingface".
DATASET_SOURCE = "huggingface"

# Evaluate every registered encoder before selecting the final three.
EMBEDDING_MODEL_KEYS = list(EMBEDDING_MODEL_SPECS)

# Embedding inference device, precision, token limit, and batch sizes.
EMBEDDING_GPU_ID = 0
EMBEDDING_USE_FP16 = True
EMBEDDING_MAX_LENGTH = 384
EMBEDDING_BATCH_SIZE = 64
RECORD_BATCH_SIZE = 64

# Local-mode record limit; None processes the complete UniEdit similarity split.
RECORD_LIMIT = None

# Replace matching result files instead of reusing them.
OVERWRITE_EXISTING_RESULT = False

# Global mode mirrors evaluation runs and seeded edit-stream sampling.
with open(EVAL_CONFIG_PATH, encoding="utf-8") as file:
    EVAL_CONFIG = yaml.safe_load(file)
GLOBAL_N_RUNS = int(EVAL_CONFIG["n_runs"])
GLOBAL_BASE_SEED = int(EVAL_CONFIG["base_seed"])
GLOBAL_BANK_SIZES = [
    int(milestone)
    for milestone in EVAL_CONFIG["eval_milestones"]
    if int(milestone) >= 100
]

# Thresholds are tuned only on UniEdit, then frozen for CounterFact.
# Options: "score", "score_margin".
GLOBAL_GATE = "score_margin"
GLOBAL_LOCALITY_FAR_TARGET = 0.01
GLOBAL_QUERY_OWNER_COUNT = 100
GLOBAL_CALIBRATION_FRACTION = 0.5
GLOBAL_POSITIVE_CALIBRATION_QUERY_LIMIT = 400
GLOBAL_POSITIVE_VALIDATION_QUERY_LIMIT = 1600
GLOBAL_LOCALITY_CALIBRATION_QUERY_LIMIT = 1000
GLOBAL_LOCALITY_VALIDATION_QUERY_LIMIT = 2000

# None evaluates every available CounterFact query in each seeded bank.
GLOBAL_TEST_POSITIVE_QUERY_LIMIT = None
GLOBAL_TEST_LOCALITY_QUERY_LIMIT = None
GLOBAL_DIAGNOSTIC_LIMIT = 5
GLOBAL_RETRIEVAL_BATCH_SIZE = 256

# Mnemo stores normalized routing keys and searches them on CPU.
GLOBAL_RETRIEVAL_DEVICE = "cpu"

# Visualization-only control for local-mode error bars.
PLOT_SHOW_STD = True

In [ ]:
# Calculate and save one JSON per enabled embedding model.
# calculate_similarity()

# Display calibration, CounterFact results, timing, memory, and overlap.
# visualize_similarity()